# ToolNode와 tools_condition

StateGraph에 ToolNode와 tools_condition을 직접 연결하면 모델의 도구 선택이 어떤 메시지 흐름으로 실행될까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 순서대로 실행합니다. API 키 값은 출력하지 않습니다.

## 패키지와 환경 준비

고정한 패키지 버전과 API 키 존재 여부만 확인합니다. 키 값 자체는 출력하지 않습니다.

## 코드를 읽기 전에

이 절은 `MessagesState`를 직접 사용하지 않고 `AgentState`를 정의합니다. `AgentState.messages`에 `add_messages` 리듀서를 붙여 새 메시지를 기존 목록에 합치고, 별도의 `trace` 목록에는 실행한 노드 이름을 기록합니다. `StateGraph(AgentState)`에 모델 노드와 `ToolNode`를 등록하고 `tools_condition`을 조건부 엣지로 연결합니다. 그래프를 `compile()`한 뒤 `invoke()`해야 노드 실행이 시작됩니다. 그래프를 그리는 코드만으로 모델이나 도구가 실행되지는 않습니다.

모델 노드는 마지막 메시지를 읽어 답 또는 `tool_calls`가 든 AIMessage를 반환합니다. 조건 함수는 마지막 AIMessage의 호출 요청 유무만 보고 `tools` 또는 종료로 보냅니다. ToolNode가 결과 ToolMessage를 추가하면 다시 모델 노드로 돌아갑니다.

In [1]:
# 패키지와 환경 준비에 필요한 연산을 실행합니다.
%pip install -q langgraph==1.2.11 langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3
import os
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / '.env').is_file():
        load_dotenv(folder / '.env')
        break
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY가 필요합니다.'
print('langgraph', version('langgraph'))
print('langchain', version('langchain'))
print('langchain-openai', version('langchain-openai'))
print('API 키 존재 여부 확인 완료')

Note: you may need to restart the kernel to use updated packages.
langgraph 1.2.11
langchain 1.4.0
langchain-openai 1.6.0
API 키 존재 여부 확인 완료


## 모델과 비용 함수 준비

실습에 사용할 모델과 토큰 사용량으로 추정 비용을 계산하는 함수를 준비합니다.

In [2]:
# 모델과 비용 함수 준비에 필요한 연산을 실행합니다.
from time import perf_counter
from typing import Any
from langchain.messages import AIMessage
from langchain_core.language_models import BaseChatModel
from langchain.chat_models import init_chat_model

MODEL_NAME = 'gpt-5.4-mini'
INPUT_USD_PER_MTOK = 0.75
OUTPUT_USD_PER_MTOK = 4.50

def usage_of(message: AIMessage) -> dict:
    usage = message.usage_metadata
    assert usage is not None, 'usage_metadata가 없어 비용을 계산할 수 없습니다.'
    assert 'input_tokens' in usage and 'output_tokens' in usage
    input_tokens = usage['input_tokens']
    output_tokens = usage['output_tokens']
    cost = input_tokens / 1_000_000 * INPUT_USD_PER_MTOK
    cost += output_tokens / 1_000_000 * OUTPUT_USD_PER_MTOK
    return {'input_tokens': input_tokens, 'output_tokens': output_tokens, 'estimated_cost_usd': cost}

def make_model() -> BaseChatModel:
    return init_chat_model(f'openai:{MODEL_NAME}', reasoning_effort='none')

## 읽기 전용 규정 도구 준비

대출·운영·연체 규정을 조회하는 lookup_policy 도구를 정의합니다.

In [3]:
# 읽기 전용 규정 도구 준비에 필요한 연산을 실행합니다.
from langchain.tools import tool

POLICIES = {
    '대출': '일반 도서는 1인 5권까지 14일 동안 대출할 수 있습니다.',
    '운영': '평일은 09:00~20:00, 토요일은 10:00~17:00에 운영합니다.',
    '연체': '연체한 일수만큼 새 대출이 제한됩니다.',
}

@tool
def lookup_policy(topic: str) -> str:
    """교육센터 자료실 규정을 주제어로 조회합니다. topic은 대출, 운영, 연체 중 하나입니다."""
    if topic not in POLICIES:
        raise ValueError(f'지원하지 않는 주제: {topic}')
    return POLICIES[topic]

TOOLS = [lookup_policy]

## 모델 노드와 실행 흔적 정의

모델 응답을 messages에 추가하고 호출 시간과 토큰 사용량을 trace에 기록합니다.

In [4]:
# 모델 노드와 실행 흔적 정의에 필요한 연산을 실행합니다.
from typing import Annotated
from typing_extensions import TypedDict
from langchain.messages import AnyMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    trace: list[str]

bound_model = make_model().bind_tools(TOOLS)

def call_model(state: AgentState) -> dict:
    response = bound_model.invoke(state['messages'])
    return {'messages': [response], 'trace': state['trace'] + ['model']}

## ToolNode 루프 직접 조립

model과 ToolNode를 등록하고 tools_condition으로 도구 실행과 종료 경로를 연결합니다.

In [5]:
# ToolNode 루프 직접 조립에 필요한 연산을 실행합니다.
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition

builder = StateGraph(AgentState)
builder.add_node('model', call_model)
builder.add_node('tools', ToolNode(TOOLS))
builder.add_edge(START, 'model')
builder.add_conditional_edges('model', tools_condition)
builder.add_edge('tools', 'model')
graph = builder.compile()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools --> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 실제 질문 실행과 전체 메시지 확인

대출 기간 질문을 실행해 HumanMessage부터 최종 AIMessage까지 전체 순서를 확인합니다.

In [6]:
# 실제 질문 실행과 전체 메시지 확인에 필요한 연산을 실행합니다.
from langchain.messages import HumanMessage

started = perf_counter()
result = graph.invoke({'messages': [HumanMessage('자료실에서 일반 도서를 며칠 빌릴 수 있나요?')], 'trace': []})
elapsed = perf_counter() - started
for index, message in enumerate(result['messages'], 1):
    calls = getattr(message, 'tool_calls', [])
    print(index, type(message).__name__, 'tool_calls=', calls, 'text=', message.text[:120])
print('trace:', result['trace'])
print('latency_sec:', round(elapsed, 3))
assert any(type(m).__name__ == 'ToolMessage' for m in result['messages'])
assert result['trace'] == ['model', 'model']

1 HumanMessage tool_calls= [] text= 자료실에서 일반 도서를 며칠 빌릴 수 있나요?
2 AIMessage tool_calls= [{'name': 'lookup_policy', 'args': {'topic': '대출'}, 'id': 'call_JcCFnakyYXFkyDOPRvDFbCr1', 'type': 'tool_call'}] text= 
3 ToolMessage tool_calls= [] text= 일반 도서는 1인 5권까지 14일 동안 대출할 수 있습니다.
4 AIMessage tool_calls= [] text= 자료실의 **일반 도서**는 **1인 5권까지, 14일 동안** 빌릴 수 있습니다.
trace: ['model', 'model']
latency_sec: 1.963


## 호출별 토큰과 비용 집계

두 번의 모델 호출에서 입력·출력 토큰과 추정 비용을 합산합니다.

In [7]:
# 호출별 토큰과 비용 집계에 필요한 연산을 실행합니다.
usages = [usage_of(m) for m in result['messages'] if type(m).__name__ == 'AIMessage']
total = {key: sum(item[key] for item in usages) for key in usages[0]}
print('calls:', usages)
print('total:', {**total, 'estimated_cost_usd': round(total['estimated_cost_usd'], 6)})
assert total['estimated_cost_usd'] < 1.0

calls: [{'input_tokens': 160, 'output_tokens': 18, 'estimated_cost_usd': 0.000201}, {'input_tokens': 210, 'output_tokens': 32, 'estimated_cost_usd': 0.0003015}]
total: {'input_tokens': 370, 'output_tokens': 50, 'estimated_cost_usd': 0.000503}


## 도구가 필요 없는 질문과 비교

인사 질문을 실행해 ToolMessage 없이 종료되는 경로와 모델 호출 횟수를 비교합니다.

In [8]:
# 도구가 필요 없는 질문과 비교에 필요한 연산을 실행합니다.
direct = graph.invoke({'messages': [HumanMessage('안녕하세요. 한 문장으로 인사해 주세요.')], 'trace': []})
print('trace:', direct['trace'])
print('message_types:', [type(m).__name__ for m in direct['messages']])
print('answer:', direct['messages'][-1].text)
assert not any(type(m).__name__ == 'ToolMessage' for m in direct['messages'])

trace: ['model']
message_types: ['HumanMessage', 'AIMessage']
answer: 안녕하세요! 반갑습니다.


## 다른 상황에 적용하기

배송 규정 질문은 모델이 바로 답할 수도 있고 최신 규정을 조회할 수도 있습니다. 한 번의 조회 뒤에도 정보가 부족하면 모델이 다른 도구를 요청해 루프가 이어질 수 있습니다. `tools_condition`은 답이 사실인지 판정하지 않고 호출 요청의 존재만 확인하므로, 품질 검사와 실행 상한은 별도로 설계해야 합니다.